#### Functional mapping Schaefer to Brainnetome

In [2]:
import re
import numpy as np
import pandas as pd
import nibabel as nib
from nibabel.affines import apply_affine

# =========================================================
# PATHS
# =========================================================
schaefer_path = "_reqs/Schaefer2018_400Parcels_7Networks_order_FSLMNI152_2mm.nii.gz"
bna_path      = "_reqs/BNA-maxprob-thr0-2mm.nii.gz"
bna_lut_path  = "_reqs/BNA_subregions.xlsx"
schaefer_lut_path = "_reqs/Schaefer2018_400Parcels_7Networks_order.txt"
smc_csv_path  = "_reqs/Schaefer_SMC_parcellation.csv"

# Outputs
output_best_raw   = "_out/Schaefer_to_BNA_best_overlap.csv"
output_all        = "_out/Schaefer_to_BNA_all_overlaps.csv"
output_best_mod   = "_out/Schaefer_to_BNA_best_overlap_mod.csv"

# =========================================================
# LOAD DATA
# =========================================================
sch_img = nib.load(schaefer_path)
sch_data = np.asarray(sch_img.get_fdata(), dtype=np.int32)
sch_aff = sch_img.affine

bna_img = nib.load(bna_path)
bna_data = np.asarray(bna_img.get_fdata(), dtype=np.int32)
bna_aff = bna_img.affine
bna_aff_inv = np.linalg.inv(bna_aff)

bna_df = pd.read_excel(bna_lut_path)
smc_df = pd.read_csv(smc_csv_path, header=None, names=["smc_order", "parcel_name"])

sch_lut = pd.read_csv(
    schaefer_lut_path,
    sep=r"\s+",
    header=None,
    usecols=[0, 1],
    names=["index", "name"],
    engine="python"
)

# =========================================================
# CLEAN BNA LUT
# =========================================================
bna_df.columns = [str(c).strip().replace("\n", " ") for c in bna_df.columns]

left_col = "Label ID.L"
right_col = "Label ID.R"
desc_col = "Unnamed: 5"   # description column

if left_col not in bna_df.columns or right_col not in bna_df.columns or desc_col not in bna_df.columns:
    raise ValueError(f"Expected columns not found. Got: {bna_df.columns.tolist()}")

bna_df[left_col] = pd.to_numeric(bna_df[left_col], errors="coerce")
bna_df[right_col] = pd.to_numeric(bna_df[right_col], errors="coerce")

left_lookup = (
    bna_df[[left_col, desc_col]]
    .dropna(subset=[left_col])
    .assign(**{left_col: lambda x: x[left_col].astype(int)})
    .set_index(left_col)[desc_col]
    .to_dict()
)

right_lookup = (
    bna_df[[right_col, desc_col]]
    .dropna(subset=[right_col])
    .assign(**{right_col: lambda x: x[right_col].astype(int)})
    .set_index(right_col)[desc_col]
    .to_dict()
)

# =========================================================
# HELPERS
# =========================================================
def normalize_schaefer_name(s):
    s = str(s).strip()
    s = s.replace("7Networks_", "7Networks")
    s = s.replace("_LH_", "LH")
    s = s.replace("_RH_", "RH")
    s = s.replace("_SomMot_", "SomMot")
    s = s.replace("_", "")
    return s

def hemisphere_from_name(name):
    name = str(name)
    if "_LH_" in name or "LHSomMot" in name:
        return "L"
    if "_RH_" in name or "RHSomMot" in name:
        return "R"
    return np.nan

def extract_somot_number(name):
    m = re.search(r"SomMot_?(\d+)", str(name))
    return int(m.group(1)) if m else np.nan

def get_parcel_voxels(label):
    return np.argwhere(sch_data == label)

def bna_desc_from_label(label_id, hemi):
    label_id = int(label_id)
    if hemi == "L":
        return left_lookup.get(label_id, np.nan)
    elif hemi == "R":
        return right_lookup.get(label_id, np.nan)
    return np.nan

def canonical_bna_motor_label(desc):
    s = str(desc)
    s_compact = re.sub(r"\s+", "", s).lower()

    if "a1/2/3tonia" in s_compact or "a123tonia" in s_compact:
        return "A1/2/3tonIa (tongue/lar)"
    if "a4tl" in s_compact:
        return "A4tl (tongue/lar)"
    if "a1/2/3ulhf" in s_compact or "a123ulhf" in s_compact:
        return "A1/2/3ulhf (UL/H/F)"
    if "a4hf" in s_compact:
        return "A4hf (H/F)"
    if "a1/2/3ll" in s_compact or "a123ll" in s_compact:
        return "A1/2/3ll (LL)"
    if "a4ll" in s_compact:
        return "A4ll (LL)"
    if "a1/2/3tru" in s_compact or "a123tru" in s_compact:
        return "A1/2/3tru (TRU)"
    if "a4ul" in s_compact:
        return "A4ul (UL)"
    if "a4t" in s_compact and "a4tl" not in s_compact:
        return "A4t (TRU)"
    if s_compact.startswith("a2,") or s_compact.startswith("a2area2") or s_compact == "a2,area2" or s_compact.startswith("a2"):
        return "A2"

    return "Other"

def coarse_func_group(canonical_label):
    # Note: we will later remap Face/Tongue -> Face Area in final outputs
    if canonical_label in ["A1/2/3tonIa (tongue/lar)", "A4tl (tongue/lar)", "A4hf (H/F)"]:
        return "Face/Tongue"
    if canonical_label in ["A1/2/3ulhf (UL/H/F)", "A4ul (UL)", "A2"]:
        return "Upper limb"
    if canonical_label in ["A1/2/3ll (LL)", "A4ll (LL)"]:
        return "Lower limb"
    if canonical_label in ["A1/2/3tru (TRU)", "A4t (TRU)"]:
        return "Trunk"
    return "Other"

def parcel_center_mni(label):
    """Return center-of-mass MNI coordinate (mean of voxel coords) for a Schaefer label."""
    vox = get_parcel_voxels(label)
    if len(vox) == 0:
        return np.nan, np.nan, np.nan
    mni_xyz = apply_affine(sch_aff, vox)
    center = mni_xyz.mean(axis=0)
    return tuple(center)

# =========================================================
# MATCH CURATED SMC LIST TO SCHAEFER LUT
# =========================================================
smc_df["parcel_name_norm"] = smc_df["parcel_name"].map(normalize_schaefer_name)
smc_df["hemi"] = smc_df["parcel_name"].map(hemisphere_from_name)
smc_df["sommot_num"] = smc_df["parcel_name"].map(extract_somot_number)

sch_lut["name_norm"] = sch_lut["name"].map(normalize_schaefer_name)

smc_match = smc_df.merge(
    sch_lut[["index", "name", "name_norm"]],
    left_on="parcel_name_norm",
    right_on="name_norm",
    how="left"
)

smc_match = smc_match.dropna(subset=["index"]).copy()
smc_match["index"] = smc_match["index"].astype(int)
smc_match["sommot_num"] = smc_match["sommot_num"].astype(int)
smc_match["smc_order"] = smc_match["smc_order"].astype(int)

# =========================================================
# MAIN BEST-OVERLAP TABLE
# =========================================================
rows_best = []
detail_rows = []

for _, r in smc_match.iterrows():
    sch_idx = int(r["index"])
    hemi = r["hemi"]
    smc_order = int(r["smc_order"])
    sommot_num = int(r["sommot_num"])
    sch_label_name = r["parcel_name"]

    parcel_voxels = get_parcel_voxels(sch_idx)
    n_vox_total = len(parcel_voxels)

    center_mni = parcel_center_mni(sch_idx)
    center_str = f"({center_mni[0]:.1f}, {center_mni[1]:.1f}, {center_mni[2]:.1f})" if not np.isnan(center_mni[0]) else "(nan, nan, nan)"

    if n_vox_total == 0:
        rows_best.append({
            "smc_order": smc_order,
            "Schaefer_parcel": sch_label_name,
            "Hemisphere": hemi,
            "sommot_num": sommot_num,
            "MNI_coordinates": center_str,
            "BNA_label": np.nan,
            "Brainnetome_label": np.nan,
            "Canonical_label": "Other",
            "Functional_class": "Other",
            "n_vox_in_parcel": 0,
            "n_vox_overlap": 0,
            "Overlap_fraction": np.nan
        })
        continue

    # Map to BNA
    mni_xyz = apply_affine(sch_aff, parcel_voxels)
    bna_vox = np.round(apply_affine(bna_aff_inv, mni_xyz)).astype(int)

    valid = (
        (bna_vox[:, 0] >= 0) & (bna_vox[:, 0] < bna_data.shape[0]) &
        (bna_vox[:, 1] >= 0) & (bna_vox[:, 1] < bna_data.shape[1]) &
        (bna_vox[:, 2] >= 0) & (bna_vox[:, 2] < bna_data.shape[2])
    )
    bna_vox = bna_vox[valid]

    if len(bna_vox) == 0:
        rows_best.append({
            "smc_order": smc_order,
            "Schaefer_parcel": sch_label_name,
            "Hemisphere": hemi,
            "sommot_num": sommot_num,
            "MNI_coordinates": center_str,
            "BNA_label": np.nan,
            "Brainnetome_label": np.nan,
            "Canonical_label": "Other",
            "Functional_class": "Other",
            "n_vox_in_parcel": n_vox_total,
            "n_vox_overlap": 0,
            "Overlap_fraction": 0.0
        })
        continue

    bna_labels = bna_data[bna_vox[:, 0], bna_vox[:, 1], bna_vox[:, 2]]
    bna_labels = bna_labels[bna_labels > 0]

    if len(bna_labels) == 0:
        rows_best.append({
            "smc_order": smc_order,
            "Schaefer_parcel": sch_label_name,
            "Hemisphere": hemi,
            "sommot_num": sommot_num,
            "MNI_coordinates": center_str,
            "BNA_label": np.nan,
            "Brainnetome_label": np.nan,
            "Canonical_label": "Other",
            "Functional_class": "Other",
            "n_vox_in_parcel": n_vox_total,
            "n_vox_overlap": 0,
            "Overlap_fraction": 0.0
        })
        continue

    # ----- all-overlaps for this parcel -----
    uniq_all, counts_all = np.unique(bna_labels, return_counts=True)
    for lab, cnt in zip(uniq_all, counts_all):
        lab = int(lab)
        cnt = int(cnt)
        frac = cnt / n_vox_total

        bna_desc = bna_desc_from_label(lab, hemi)
        canonical_label = canonical_bna_motor_label(bna_desc)
        func_group = coarse_func_group(canonical_label)

        detail_rows.append({
            "smc_order": smc_order,
            "Schaefer_parcel": sch_label_name,
            "Hemisphere": hemi,
            "sommot_num": sommot_num,
            "BNA_label": lab,
            "Brainnetome_label": bna_desc,
            "Canonical_label": canonical_label,
            "Functional_class": func_group,
            "n_vox_in_parcel": n_vox_total,
            "n_vox_overlap": cnt,
            "Overlap_fraction": frac
        })

    # ----- best-overlap -----
    best_i = np.argmax(counts_all)
    best_label = int(uniq_all[best_i])
    best_count = int(counts_all[best_i])
    overlap_fraction = best_count / n_vox_total

    bna_desc = bna_desc_from_label(best_label, hemi)
    canonical_label = canonical_bna_motor_label(bna_desc)
    func_group = coarse_func_group(canonical_label)

    rows_best.append({
        "smc_order": smc_order,
        "Schaefer_parcel": sch_label_name,
        "Hemisphere": hemi,
        "sommot_num": sommot_num,
        "MNI_coordinates": center_str,
        "BNA_label": best_label,
        "Brainnetome_label": bna_desc,
        "Canonical_label": canonical_label,
        "Functional_class": func_group,
        "n_vox_in_parcel": n_vox_total,
        "n_vox_overlap": best_count,
        "Overlap_fraction": overlap_fraction
    })

# Build DataFrames
df_best_raw = pd.DataFrame(rows_best).sort_values(["Hemisphere", "sommot_num"]).reset_index(drop=True)
df_all = pd.DataFrame(detail_rows).sort_values(
    ["Hemisphere", "sommot_num", "Overlap_fraction"],
    ascending=[True, True, False]
).reset_index(drop=True)

# =========================================================
# APPLY MANUAL RULES FOR BODERLINE REGIONS (MODIFIED BEST TABLE) - Manually Verified
# =========================================================
df_mod = df_best_raw.copy()

# Convenience masks on df_mod
mask_A40rv = df_mod["Brainnetome_label"] == "A40rv, rostroventral area 40(PFop)"
df_mod.loc[mask_A40rv, "Canonical_label"] = "A40rv"
df_mod.loc[mask_A40rv, "Functional_class"] = "Face/Tongue"

mask_A6m = df_mod["Brainnetome_label"] == "A6m, medial area 6"
df_mod.loc[mask_A6m, "Canonical_label"] = "A6m"
df_mod.loc[mask_A6m, "Functional_class"] = "Lower limb"

mask_A7pc = df_mod["Brainnetome_label"] == "A7pc, postcentral area 7"
df_mod.loc[mask_A7pc, "Canonical_label"] = "A7pc"
df_mod.loc[mask_A7pc, "Functional_class"] = "Trunk"

mask_A6cdl = df_mod["Brainnetome_label"] == "A6cdl, caudal dorsolateral area 6"
df_mod.loc[mask_A6cdl, "Canonical_label"] = "A6cdl"
df_mod.loc[mask_A6cdl, "Functional_class"] = "Upper limb"

mask_A6dl = df_mod["Brainnetome_label"] == "A6dl, dorsolateral area 6"
df_mod.loc[mask_A6dl & (df_mod["Hemisphere"] == "L"), "Canonical_label"] = "A6dl"
df_mod.loc[mask_A6dl & (df_mod["Hemisphere"] == "L"), "Functional_class"] = "Trunk"
df_mod.loc[mask_A6dl & (df_mod["Hemisphere"] == "R"), "Canonical_label"] = "A6dl"
df_mod.loc[mask_A6dl & (df_mod["Hemisphere"] == "R"), "Functional_class"] = "Upper limb"

mask_A23c = df_mod["Brainnetome_label"] == "A23c, caudal area 23"
df_mod.loc[mask_A23c, "Brainnetome_label"] = "A6m, medial area 6"
df_mod.loc[mask_A23c, "Canonical_label"] = "A6m"
df_mod.loc[mask_A23c, "Functional_class"] = "Lower limb"

mask_A22c = df_mod["Brainnetome_label"] == "A22c, caudal area 22"
df_mod.loc[mask_A22c, "Canonical_label"] = "Other"
df_mod.loc[mask_A22c, "Functional_class"] = "Other"

mask_A123ulhf = df_mod["Brainnetome_label"] == "A1/2/3ulhf, area 1/2/3(upper limb, head and face region)"
df_mod.loc[mask_A123ulhf & (df_mod["Schaefer_parcel"] == "7Networks_RH_SomMot_16"), "Canonical_label"] = "A1/2/3ulhf (UL/H/F)"
df_mod.loc[mask_A123ulhf, "Functional_class"] = "Face/Tongue"

# =========================================================
# FINAL COLUMN SELECTION / RENAMING
# =========================================================
# In best and best_mod, expose only the requested columns
keep_cols = [
    "Schaefer_parcel",
    "Hemisphere",
    "MNI_coordinates",
    "BNA_label",          # <- keep this
    "Brainnetome_label",
    "Canonical_label",
    "Overlap_fraction",
    "Functional_class"
]

df_best = df_best_raw[keep_cols].copy()
df_best_mod = df_mod[keep_cols].copy()

# Remap Functional_class Face/Tongue -> Face Area
for df_tmp in (df_best, df_best_mod):
    df_tmp["Functional_class"] = df_tmp["Functional_class"].replace({"Face/Tongue": "Face Area"})

# =========================================================
# SAVE
# =========================================================
df_best.to_csv(output_best_raw, index=False)
df_all.to_csv(output_all, index=False)
df_best_mod.to_csv(output_best_mod, index=False)

print("Saved:")
print("  BEST (raw):", output_best_raw)
print("  ALL overlaps:", output_all)
print("  BEST (modified):", output_best_mod)

# Summary statistics of overlap (after manual rules; same as before for overlaps)
mean_ov = df_best_mod["Overlap_fraction"].mean()
std_ov = df_best_mod["Overlap_fraction"].std()
print("Overlap_fraction mean:", mean_ov)
print("Overlap_fraction std :", std_ov)

Saved:
  BEST (raw): /media/RCPNAS/Data2/Ekansh/templates/Gradients/Schaefer_to_BNA_best_overlap.csv
  ALL overlaps: /media/RCPNAS/Data2/Ekansh/templates/Gradients/Schaefer_to_BNA_all_overlaps.csv
  BEST (modified): /media/RCPNAS/Data2/Ekansh/templates/Gradients/Schaefer_to_BNA_best_overlap_mod.csv
Overlap_fraction mean: 0.6127734699694339
Overlap_fraction std : 0.14937643041897594
